In [2]:
# Notebook 01, cell 1 — setup: make src/ importable, load the tested code, record versions
import sys
from pathlib import Path

# Walk up from the notebook's folder until we find the project root (the folder containing src/)
project_root = Path.cwd()
while not (project_root / "src").is_dir() and project_root != project_root.parent:
    project_root = project_root.parent
if not (project_root / "src").is_dir():
    raise RuntimeError(f"Could not find the src folder above {Path.cwd()}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import scipy
import pandas as pd
import matplotlib
import torch

from src.pricing import black76_price, black76_delta, black76_gamma, black76_vega, black76_theta
from src.simulate import simulate_gbm_paths, mc_price_european, SEED
from src.hedging import hedge_pnl

print(f"Project root: {project_root}")
print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | scipy {scipy.__version__} | "
      f"pandas {pd.__version__} | matplotlib {matplotlib.__version__} | torch {torch.__version__}")
print(f"Seed: {SEED}")
print("All project imports OK")

Project root: /Users/nonu/Desktop/Projects/deep-hedging
Python 3.13.5 | numpy 2.5.4 | scipy 1.18.1 | pandas 3.0.6 | matplotlib 3.11.2 | torch 2.14.1
Seed: 42
All project imports OK


In [3]:
# Notebook 01, cell 2 — Gate test 1: put-call parity across a grid of strikes, maturities, vols and rates
F = 100.0
strikes = np.linspace(50.0, 150.0, 21)
maturities = np.array([1 / 252, 30 / 252, 0.5, 2.0])
vols = np.array([0.10, 0.30, 0.60, 1.00])
rates = np.array([0.0, 0.05])

# Every combination of strike x maturity x vol x rate, priced in one vectorised call
K, T, sig, r = np.meshgrid(strikes, maturities, vols, rates, indexing="ij")

calls = black76_price(F, K, T, r, sig, "call")
puts = black76_price(F, K, T, r, sig, "put")
parity_gap = np.abs((calls - puts) - np.exp(-r * T) * (F - K))

call_deltas = black76_delta(F, K, T, r, sig, "call")
put_deltas = black76_delta(F, K, T, r, sig, "put")
delta_gap = np.abs((call_deltas - put_deltas) - np.exp(-r * T))

TOL_PARITY = 1e-12
worst = np.unravel_index(np.argmax(parity_gap), parity_gap.shape)

print(f"Combinations tested: {parity_gap.size}")
print(f"Max price parity gap: {parity_gap.max():.2e}   (tolerance {TOL_PARITY:.0e})")
print(f"Max delta parity gap: {delta_gap.max():.2e}   (tolerance {TOL_PARITY:.0e})")
print(f"Worst case: K={K[worst]:.0f}, T={T[worst]:.4f}y, vol={sig[worst]:.0%}, r={r[worst]:.0%}")

gate_parity = bool(parity_gap.max() < TOL_PARITY and delta_gap.max() < TOL_PARITY)
print("\nGATE 1 (put-call parity):", "PASS" if gate_parity else "FAIL")

Combinations tested: 672
Max price parity gap: 2.84e-14   (tolerance 1e-12)
Max delta parity gap: 1.11e-16   (tolerance 1e-12)
Worst case: K=130, T=2.0000y, vol=100%, r=0%

GATE 1 (put-call parity): PASS


In [4]:
# Notebook 01, cell 3 — Gate test 2: Monte Carlo vs closed form across strikes, calls/puts, two maturities
from scipy.stats import norm

F0_mc, r_mc, sigma_mc = 100.0, 0.05, 0.30
strikes_mc = np.arange(80.0, 121.0, 5.0)        # 80, 85, ..., 120  (9 strikes)
maturities_mc = [30 / 252, 0.5]                  # one month, six months
N_PATHS_MC = 200_000

rows = []
test_id = 0
for T_mc in maturities_mc:
    for K_mc in strikes_mc:
        for opt in ("call", "put"):
            seed = SEED + 1000 + test_id         # independent draws for every test
            mc, se = mc_price_european(F0_mc, K_mc, T_mc, r_mc, sigma_mc, opt,
                                       n_paths=N_PATHS_MC, seed=seed)
            cf = float(black76_price(F0_mc, K_mc, T_mc, r_mc, sigma_mc, opt))
            rows.append({"T_years": round(T_mc, 4), "K": K_mc, "type": opt,
                         "closed_form": cf, "monte_carlo": mc, "std_err": se,
                         "z": (mc - cf) / se})
            test_id += 1

results_mc = pd.DataFrame(rows)
n_tests = len(results_mc)
abs_z = results_mc["z"].abs()

FAMILY_ALPHA = 0.01                               # 1% chance a correct pricer fails the whole gate
z_threshold = norm.ppf(1 - FAMILY_ALPHA / (2 * n_tests))

print(f"{n_tests} tests, {N_PATHS_MC:,} paths each, r = {r_mc:.0%}, vol = {sigma_mc:.0%}")
print(f"Share within 1 SE: {(abs_z < 1).mean():.0%}   (normal noise: ~68%)")
print(f"Share within 2 SE: {(abs_z < 2).mean():.0%}   (normal noise: ~95%)")
print(f"Count beyond 3 SE: {(abs_z > 3).sum()}      (expected by chance: {n_tests * 0.0027:.2f})")
print(f"Max |z| = {abs_z.max():.2f}   vs Bonferroni threshold {z_threshold:.2f}")

print("\nFive largest deviations:")
print(results_mc.reindex(abs_z.sort_values(ascending=False).index).head(5)
      .round(4).to_string(index=False))

gate_mc = bool(abs_z.max() < z_threshold)
print("\nGATE 2 (Monte Carlo within error):", "PASS" if gate_mc else "FAIL")

36 tests, 200,000 paths each, r = 5%, vol = 30%
Share within 1 SE: 64%   (normal noise: ~68%)
Share within 2 SE: 100%   (normal noise: ~95%)
Count beyond 3 SE: 0      (expected by chance: 0.10)
Max |z| = 1.91   vs Bonferroni threshold 3.64

Five largest deviations:
 T_years     K type  closed_form  monte_carlo  std_err       z
   0.500 120.0  put      21.9482      22.0183   0.0368  1.9057
   0.500  80.0 call      20.8964      20.8171   0.0425 -1.8684
   0.500  85.0  put       2.4251       2.4477   0.0124  1.8226
   0.119 110.0  put      10.9830      11.0170   0.0191  1.7797
   0.500  80.0  put       1.3902       1.4051   0.0091  1.6425

GATE 2 (Monte Carlo within error): PASS


In [5]:
# Notebook 01, cell 4 — Gate test 3 (Hull textbook example) and Phase 0 gate summary, saved to outputs/
import json
from datetime import datetime, timezone

# Gate test 3: Hull, Black's model futures option. F = K = 20, r = 9%, T = 4 months, vol = 25%.
hull_inputs = dict(F=20.0, K=20.0, T=4 / 12, r=0.09, sigma=0.25)
HULL_PUBLISHED = 1.12          # Hull reports the price to 2 decimals
HULL_TOL = 0.005               # so our price must round to 1.12

hull_call = float(black76_price(**hull_inputs, option_type="call"))
hull_put = float(black76_price(**hull_inputs, option_type="put"))
gate_hull = bool(abs(hull_call - HULL_PUBLISHED) <= HULL_TOL and abs(hull_put - HULL_PUBLISHED) <= HULL_TOL)

print(f"Hull example: call = {hull_call:.4f}, put = {hull_put:.4f}, published = {HULL_PUBLISHED}")
print("GATE 3 (textbook example):", "PASS" if gate_hull else "FAIL")

# Phase 0 summary
summary = pd.DataFrame([
    {"gate": "1. Put-call parity (672 combinations)", "result": "PASS" if gate_parity else "FAIL",
     "evidence": f"max gap {parity_gap.max():.1e} vs tolerance {TOL_PARITY:.0e}"},
    {"gate": "2. Monte Carlo vs closed form (36 tests)", "result": "PASS" if gate_mc else "FAIL",
     "evidence": f"max |z| {abs_z.max():.2f} vs Bonferroni threshold {z_threshold:.2f}"},
    {"gate": "3. Hull textbook example", "result": "PASS" if gate_hull else "FAIL",
     "evidence": f"call {hull_call:.4f}, put {hull_put:.4f} vs published {HULL_PUBLISHED}"},
])
phase0_pass = gate_parity and gate_mc and gate_hull
print("\n" + summary.to_string(index=False))
print("\nPHASE 0 GATE:", "PASS" if phase0_pass else "FAIL")

# Save the record next to the outputs (reproducibility rule)
out_dir = project_root / "outputs" / "phase0"
out_dir.mkdir(parents=True, exist_ok=True)
record = {
    "phase": 0,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "seed": SEED,
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "scipy": scipy.__version__,
                 "pandas": pd.__version__, "matplotlib": matplotlib.__version__, "torch": torch.__version__},
    "gate_parity": {"pass": gate_parity, "max_price_gap": float(parity_gap.max()),
                    "max_delta_gap": float(delta_gap.max()), "tolerance": TOL_PARITY,
                    "n_combinations": int(parity_gap.size)},
    "gate_monte_carlo": {"pass": gate_mc, "max_abs_z": float(abs_z.max()), "z_threshold": float(z_threshold),
                         "family_alpha": FAMILY_ALPHA, "n_tests": int(n_tests), "paths_per_test": N_PATHS_MC},
    "gate_hull": {"pass": gate_hull, "call": hull_call, "put": hull_put,
                  "published": HULL_PUBLISHED, "tolerance": HULL_TOL},
    "phase0_pass": bool(phase0_pass),
}
with open(out_dir / "gate_results.json", "w") as f:
    json.dump(record, f, indent=2)
print(f"\nSaved: {out_dir / 'gate_results.json'}")

Hull example: call = 1.1166, put = 1.1166, published = 1.12
GATE 3 (textbook example): PASS

                                    gate result                                  evidence
   1. Put-call parity (672 combinations)   PASS        max gap 2.8e-14 vs tolerance 1e-12
2. Monte Carlo vs closed form (36 tests)   PASS max |z| 1.91 vs Bonferroni threshold 3.64
                3. Hull textbook example   PASS call 1.1166, put 1.1166 vs published 1.12

PHASE 0 GATE: PASS

Saved: /Users/nonu/Desktop/Projects/deep-hedging/outputs/phase0/gate_results.json
